In [1]:
# If running in a fresh environment, uncomment the next line:
# %pip install requests pandas openpyxl urllib3 --quiet

import json
import re
import time
from datetime import datetime, timezone

import requests
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter

print("Imports OK")


Imports OK


In [5]:
protein_name = "ITK"          # <-- CHANGE THIS to run for a different protein
organism_id  = "9606"         # NCBI taxon ID filter for the UniProt search, default = human.
                               # Set to None to search across all organisms.
preferred_reviewed_only_first = True   # Prefer Swiss-Prot (reviewed) entries first.

OUTPUT_XLSX = "Protein_PDB_Data.xlsx"


In [7]:
SESSION = requests.Session()
_retry = Retry(
    total=5,
    backoff_factor=1.5,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)
SESSION.mount("https://", HTTPAdapter(max_retries=_retry))
SESSION.headers.update({"User-Agent": "protein-pdb-excel-pipeline/1.0 (research use)"})

REQUEST_TIMEOUT = 30      # seconds
POLITE_DELAY = 0.15       # seconds between calls to be a good API citizen

NA = "N/A"

def safe_get_json(url, params=None, context=""):
    """GET a URL and return parsed JSON, or None on any failure. Never raises.
    Logs failures into FAILURE_LOG for the final processing summary."""
    try:
        resp = SESSION.get(url, params=params, timeout=REQUEST_TIMEOUT)
        time.sleep(POLITE_DELAY)
        if resp.status_code == 404:
            FAILURE_LOG.append({"context": context, "url": resp.url, "reason": "404 Not Found"})
            return None
        resp.raise_for_status()
        return resp.json()
    except requests.exceptions.RequestException as e:
        FAILURE_LOG.append({"context": context, "url": url, "reason": f"{type(e).__name__}: {e}"})
        return None
    except json.JSONDecodeError as e:
        FAILURE_LOG.append({"context": context, "url": url, "reason": f"Malformed JSON: {e}"})
        return None

FAILURE_LOG = []   # list of dicts: {context, url, reason} -- printed in the final summary

def deep_find(obj, key_patterns, path=""):
    """Recursively search a JSON-like structure (dict/list) for the FIRST key whose name
    matches any of the given regex patterns (case-insensitive). Returns (value, path) or
    (None, None). This is a transparent, narrowly-scoped fallback -- it is only ever called
    with patterns specific to the single quantity being extracted (e.g. only 'clashscore'
    patterns when looking for Clashscore), never a generic catch-all, so it cannot silently
    pull in an unrelated value."""
    compiled = [re.compile(p, re.IGNORECASE) for p in key_patterns]
    if isinstance(obj, dict):
        for k, v in obj.items():
            newpath = f"{path}.{k}" if path else k
            if any(c.search(k) for c in compiled) and not isinstance(v, (dict, list)):
                return v, newpath
        for k, v in obj.items():
            newpath = f"{path}.{k}" if path else k
            found, foundpath = deep_find(v, key_patterns, newpath)
            if found is not None:
                return found, foundpath
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            found, foundpath = deep_find(item, key_patterns, f"{path}[{i}]")
            if found is not None:
                return found, foundpath
    return None, None

print("Helpers defined")


Helpers defined


In [9]:
UNIPROT_BASE = "https://rest.uniprot.org/uniprotkb"

def uniprot_search(name, organism_id=None):
    query = f'(gene:{name})'
    if organism_id:
        query += f' AND (organism_id:{organism_id})'
    params = {
        "query": query,
        "fields": "accession,id,gene_names,protein_name,organism_name,reviewed,xref_pdb",
        "format": "json",
        "size": 25,
    }
    data = safe_get_json(f"{UNIPROT_BASE}/search", params=params, context="UniProt search")
    if not data:
        return []
    return data.get("results", [])

candidates = uniprot_search(protein_name, organism_id)

# Broaden the search (drop organism filter) if nothing came back with it applied.
if not candidates and organism_id:
    print(f"No hits restricted to organism_id={organism_id}; retrying without organism filter.")
    candidates = uniprot_search(protein_name, organism_id=None)

if not candidates:
    raise RuntimeError(
        f"UniProt search returned no candidates for protein_name='{protein_name}'. "
        f"Check the spelling / gene symbol, or search manually at https://www.uniprot.org/"
    )

def is_reviewed(entry):
    return entry.get("entryType", "").startswith("UniProtKB reviewed")

def gene_names_of(entry):
    names = []
    for g in entry.get("genes", []) or []:
        gn = g.get("geneName", {}).get("value")
        if gn:
            names.append(gn)
        for syn in g.get("synonyms", []) or []:
            v = syn.get("value")
            if v:
                names.append(v)
    return names

reviewed_candidates = [c for c in candidates if is_reviewed(c)] if preferred_reviewed_only_first else candidates
pool = reviewed_candidates if reviewed_candidates else candidates

exact_gene_matches = [c for c in pool if protein_name.upper() in [g.upper() for g in gene_names_of(c)]]

print(f"Total UniProt candidates: {len(candidates)}  |  Reviewed: {len(reviewed_candidates)}  "
      f"|  Exact gene-name matches (within chosen pool): {len(exact_gene_matches)}")

chosen_entry = None
ambiguous = False

if len(exact_gene_matches) == 1:
    chosen_entry = exact_gene_matches[0]
elif len(exact_gene_matches) > 1:
    ambiguous = True
elif len(pool) == 1:
    chosen_entry = pool[0]
else:
    ambiguous = True

if ambiguous or chosen_entry is None:
    print("\n*** AMBIGUOUS or UNRESOLVED UniProt match -- not auto-selecting. ***")
    print("Candidates (accession | reviewed? | gene names | protein name):")
    for c in (exact_gene_matches or pool):
        acc = c.get("primaryAccession")
        rev = is_reviewed(c)
        genes = ", ".join(gene_names_of(c)) or "(none)"
        pname = c.get("proteinDescription", {}).get("recommendedName", {}).get("fullName", {}).get("value", "")
        print(f"  {acc} | reviewed={rev} | genes=[{genes}] | {pname}")
    raise RuntimeError(
        "Multiple (or zero) plausible UniProt entries found and the correct one could not be "
        "determined automatically. Inspect the printed candidates above and set:\n"
        "    chosen_accession = '<the correct accession>'\n"
        "then re-run from the next cell onward instead of from the top."
    )

chosen_accession = chosen_entry.get("primaryAccession")
print(f"\nSelected UniProt entry: {chosen_accession}  (reviewed={is_reviewed(chosen_entry)})")


Total UniProt candidates: 7  |  Reviewed: 1  |  Exact gene-name matches (within chosen pool): 1

Selected UniProt entry: Q08881  (reviewed=True)


In [11]:
# Fetch the FULL UniProt entry (JSON) for the chosen accession -- this is the authoritative
# record used for the PDB cross-references in Section 4, and for the display name in Column 1.
uniprot_entry = safe_get_json(f"{UNIPROT_BASE}/{chosen_accession}.json",
                               context=f"UniProt full entry {chosen_accession}")
if uniprot_entry is None:
    raise RuntimeError(f"Could not fetch the full UniProt entry for {chosen_accession}.")

recommended_name = (
    uniprot_entry.get("proteinDescription", {})
                 .get("recommendedName", {})
                 .get("fullName", {})
                 .get("value", protein_name)
)
protein_name_column_value = f"{protein_name} ({chosen_accession})"
print("Column 1 value will be:", protein_name_column_value)
print("UniProt recommended protein name:", recommended_name)
UNIPROT_ENTRY_URL = f"https://www.uniprot.org/uniprotkb/{chosen_accession}/entry"


Column 1 value will be: ITK (Q08881)
UniProt recommended protein name: Tyrosine-protein kinase ITK/TSK


In [13]:
def extract_pdb_xrefs(entry_json):
    pdb_ids = []
    uniprot_reported = {}   # pdb_id -> {"Method":..., "Resolution":..., "Chains":...}
    for xref in entry_json.get("uniProtKBCrossReferences", []):
        if xref.get("database") == "PDB":
            pdb_id = xref.get("id")
            if not pdb_id:
                continue
            props = {p["key"]: p["value"] for p in xref.get("properties", []) if "key" in p}
            pdb_ids.append(pdb_id)
            uniprot_reported[pdb_id] = props
    # de-duplicate, preserve order
    pdb_ids = list(dict.fromkeys(pdb_ids))
    return pdb_ids, uniprot_reported

pdb_ids, uniprot_reported_props = extract_pdb_xrefs(uniprot_entry)

print(f"Total PDB IDs found for {protein_name_column_value}: {len(pdb_ids)}")
print(pdb_ids)

if not pdb_ids:
    print("WARNING: UniProt reports zero associated PDB structures for this entry. "
          "The Excel workbook will still be generated but will contain no structural rows.")


Total PDB IDs found for ITK (Q08881): 37
['1SM2', '1SNU', '1SNX', '2E6I', '2LMJ', '2YUQ', '3MIY', '3MJ1', '3MJ2', '3QGW', '3QGY', '3T9T', '3V5J', '3V5L', '3V8T', '3V8W', '4HCT', '4HCU', '4HCV', '4KIO', '4L7S', '4M0Y', '4M0Z', '4M12', '4M13', '4M14', '4M15', '4MF0', '4MF1', '4PP9', '4PPA', '4PPB', '4PPC', '4PQN', '4QD6', '4RFM', '9NWX']


In [15]:
RCSB_ENTRY_URL = "https://data.rcsb.org/rest/v1/core/entry/{pdb_id}"

def rcsb_entry_link(pdb_id):
    return f"https://www.rcsb.org/structure/{pdb_id}"

entry_json_by_pdb = {}
for pdb_id in pdb_ids:
    data = safe_get_json(RCSB_ENTRY_URL.format(pdb_id=pdb_id), context=f"RCSB entry {pdb_id}")
    entry_json_by_pdb[pdb_id] = data   # may be None -- handled downstream, does not stop the loop

print(f"Fetched RCSB entry records for {sum(v is not None for v in entry_json_by_pdb.values())} "
      f"of {len(pdb_ids)} PDB IDs.")


Fetched RCSB entry records for 37 of 37 PDB IDs.


In [17]:
# --- Sanity dump: inspect ONE raw entry JSON before trusting the parsing rules below. ---
# Run this after Section 5 the first time you use the notebook for a new protein, or whenever
# RCSB appears to have changed something. It is not required on every run.
_sample_id = next((p for p, v in entry_json_by_pdb.items() if v), None)
if _sample_id:
    print(f"Sample raw entry JSON top-level keys for {_sample_id}:")
    print(sorted(entry_json_by_pdb[_sample_id].keys()))
else:
    print("No successful RCSB entry fetches to sample.")


Sample raw entry JSON top-level keys for 1SM2:
['audit_author', 'cell', 'citation', 'database_2', 'diffrn', 'diffrn_detector', 'diffrn_radiation', 'diffrn_source', 'entry', 'exptl', 'exptl_crystal', 'exptl_crystal_grow', 'pdbx_audit_revision_category', 'pdbx_audit_revision_details', 'pdbx_audit_revision_group', 'pdbx_audit_revision_history', 'pdbx_audit_revision_item', 'pdbx_database_status', 'pdbx_vrpt_summary', 'pdbx_vrpt_summary_geometry', 'rcsb_accession_info', 'rcsb_binding_affinity', 'rcsb_entry_container_identifiers', 'rcsb_entry_info', 'rcsb_id', 'rcsb_primary_citation', 'refine', 'refine_hist', 'refine_ls_restr', 'reflns', 'reflns_shell', 'software', 'struct', 'struct_keywords', 'symmetry']


In [19]:
def get_method(entry):
    if not entry:
        return NA, "not_found"
    exptl = entry.get("exptl") or []
    methods = [e.get("method") for e in exptl if e.get("method")]
    if methods:
        return "; ".join(dict.fromkeys(methods)), "primary:exptl.method"
    val, path = deep_find(entry, [r"^method$"])
    return (val, f"fallback:{path}") if val else (NA, "not_found")

def get_resolution(entry):
    if not entry:
        return NA, "not_found"
    info = entry.get("rcsb_entry_info") or {}
    res = info.get("resolution_combined")
    if res:
        # resolution_combined is a list (usually one value); keep as a number, joined if >1
        if len(res) == 1:
            return res[0], "primary:rcsb_entry_info.resolution_combined"
        return "; ".join(str(r) for r in res), "primary:rcsb_entry_info.resolution_combined"
    val, path = deep_find(entry, [r"resolution"])
    return (val, f"fallback:{path}") if val is not None else (NA, "not_applicable_for_method")

print("Method/resolution extractors defined")


Method/resolution extractors defined


In [21]:
RCSB_POLYMER_ENTITY_URL = "https://data.rcsb.org/rest/v1/core/polymer_entity/{pdb_id}/{entity_id}"
RCSB_POLYMER_INSTANCE_URL = "https://data.rcsb.org/rest/v1/core/polymer_entity_instance/{pdb_id}/{asym_id}"

def get_matching_polymer_entity(pdb_id, entry, target_accession):
    """Return (entity_json, entity_id, match_method) for the polymer entity that represents the
    target protein in this PDB entry."""
    ids = (entry.get("rcsb_entry_container_identifiers") or {}).get("polymer_entity_ids") or []
    entities = []
    for eid in ids:
        ejson = safe_get_json(RCSB_POLYMER_ENTITY_URL.format(pdb_id=pdb_id, entity_id=eid),
                               context=f"RCSB polymer_entity {pdb_id}_{eid}")
        if ejson:
            entities.append((eid, ejson))

    # 1) match by UniProt accession
    for eid, ejson in entities:
        uids = (ejson.get("rcsb_polymer_entity_container_identifiers") or {}).get("uniprot_ids") or []
        if target_accession in uids:
            return ejson, eid, "matched_by_uniprot_accession"

    # 2) fallback: first entity explicitly typed as Protein
    for eid, ejson in entities:
        poly_type = (ejson.get("entity_poly") or {}).get("rcsb_entity_polymer_type", "")
        if poly_type == "Protein":
            return ejson, eid, "fallback_first_protein_entity"

    return None, None, "no_polymer_entity_found"

def get_residue_counts(pdb_id, entry, target_accession):
    if not entry:
        return NA, NA, NA, "no_entry_data"

    entity_json, entity_id, match_method = get_matching_polymer_entity(pdb_id, entry, target_accession)
    if entity_json is None:
        return NA, NA, NA, match_method

    seq = (entity_json.get("entity_poly") or {}).get("pdbx_seq_one_letter_length", "") or ""
    deposited = len(seq) if seq else None
    if deposited is None:
        val, path = deep_find(entity_json, [r"seq_one_letter"])
        deposited = len(val) if isinstance(val, str) else None
    if deposited is None:
        return NA, NA, NA, f"{match_method}; deposited_length_not_found"

    asym_ids = (entity_json.get("rcsb_polymer_entity_container_identifiers") or {}).get("asym_ids") or []
    best_modeled = None
    best_asym = None
    for asym_id in asym_ids:
        inst = safe_get_json(RCSB_POLYMER_INSTANCE_URL.format(pdb_id=pdb_id, asym_id=asym_id),
                              context=f"RCSB polymer_entity_instance {pdb_id}.{asym_id}")
        if not inst:
            continue
        unobserved = 0
        for feat in inst.get("rcsb_polymer_instance_feature_summary") or []:
            if feat.get("type") == "UNOBSERVED_RESIDUE_XYZ":
                unobserved = feat.get("count", 0) or 0
                break
        modeled = deposited - unobserved
        if best_modeled is None or modeled > best_modeled:
            best_modeled = modeled
            best_asym = asym_id

    if best_modeled is None:
        return deposited, NA, NA, f"{match_method}; modeled_count_unavailable_for_all_instances"

    diff = deposited - best_modeled
    detail = f"{match_method}; representative_chain={best_asym}"
    return deposited, best_modeled, diff, detail

print("Residue-count extractor defined")


Residue-count extractor defined


In [23]:
def get_r_values_dcc(entry):
    if not entry:
        return NA, NA, NA, "no_entry_data"
    refine_list = entry.get("refine") or []
    r_free_dcc, r_work_dcc = None, None
    for r in refine_list:
        if r_free_dcc is None and r.get("pdbx_DCC_R_free") is not None:
            r_free_dcc = r.get("pdbx_DCC_R_free")
        if r_work_dcc is None and r.get("pdbx_DCC_R_work") is not None:
            r_work_dcc = r.get("pdbx_DCC_R_work")

    note = "primary:refine.pdbx_DCC_R_free/work"
    if r_free_dcc is None:
        r_free_dcc, path = deep_find(entry, [r"DCC.*R.?free", r"R.?free.*DCC"])
        note = f"fallback:{path}" if r_free_dcc is not None else "DCC_R_free_not_available"
    if r_work_dcc is None:
        r_work_dcc, path2 = deep_find(entry, [r"DCC.*R.?work", r"R.?work.*DCC"])
        if r_work_dcc is not None:
            note += f" | fallback_work:{path2}"
        else:
            note += " | DCC_R_work_not_available"

    if r_free_dcc is None:
        r_free_dcc = NA
    if r_work_dcc is None:
        r_work_dcc = NA

    if isinstance(r_free_dcc, (int, float)) and isinstance(r_work_dcc, (int, float)):
        diff = r_free_dcc - r_work_dcc
    else:
        diff = NA
    return r_free_dcc, r_work_dcc, diff, note

print("R-value (DCC) extractor defined")


R-value (DCC) extractor defined


In [25]:
def _first_present(d, keys):
    for k in keys:
        if isinstance(d, dict) and k in d and d[k] is not None:
            return d[k], f"primary:{k}"
    return None, None

def get_validation_metrics(entry):
    """Returns dict with r_free, clashscore, rama_outliers, sidechain_outliers, rsrz_outliers,
    avg_b_all_atoms, plus a 'note' field documenting how each was obtained."""
    result = {"r_free": NA, "clashscore": NA, "rama_outliers": NA,
              "sidechain_outliers": NA, "rsrz_outliers": NA, "avg_b_all_atoms": NA}
    notes = {}

    if not entry:
        for k in result:
            notes[k] = "no_entry_data"
        result["_notes"] = notes
        return result

    vrpt_list = entry.get("pdbx_vrpt_summary")
    vrpt = None
    if isinstance(vrpt_list, list) and vrpt_list:
        vrpt = vrpt_list[0]
    elif isinstance(vrpt_list, dict):
        vrpt = vrpt_list

    field_map = {
        "r_free":            (["PDB_R_free", "Rfree", "R_free"], [r"^r.?free$", r"vrpt.*r.?free"]),
        "clashscore":        (["clashscore"], [r"clash.?score"]),
        "rama_outliers":     (["percent_rama_outliers", "percent_ramachandran_outliers"], [r"ramachandran.*outlier", r"rama.*outlier"]),
        "sidechain_outliers":(["percent_rota_outliers", "percent_sidechain_outliers", "percent_rotamer_outliers"], [r"(sidechain|rotamer).*outlier"]),
        "rsrz_outliers":     (["percent_RSRZ_outliers", "percent_rsrz_outliers"], [r"rsrz.*outlier"]),
        "avg_b_all_atoms":   (["average_b_property", "average_B_all_atoms", "PDB_average_B"], [r"average.?b.*all.?atom", r"^average_b"]),
    }

    for key, (primary_keys, fallback_patterns) in field_map.items():
        val, note = (None, None)
        if vrpt is not None:
            val, note = _first_present(vrpt, primary_keys)
        if val is None:
            val, path = deep_find(entry, fallback_patterns)
            note = f"fallback:{path}" if val is not None else "not_available_in_validation_report"
        result[key] = val if val is not None else NA
        notes[key] = note

    result["_notes"] = notes
    return result

print("wwPDB validation-metric extractor defined")


wwPDB validation-metric extractor defined


In [27]:
RCSB_NONPOLY_URL = "https://data.rcsb.org/rest/v1/core/nonpolymer_entity/{pdb_id}/{entity_id}"

NON_INFORMATIVE_COMP_IDS = {
    "HOH", "NA", "CL", "K", "MG", "CA", "ZN", "SO4", "PO4", "GOL", "EDO", "PEG",
    "ACT", "TRS", "IPA", "DMS", "FMT", "BME", "MPD", "UNX", "PG4", "1PE", "EPE",
    "MES", "IMD", "CO", "MN", "NI", "CU", "BR", "IOD", "NH4",
}

def get_ligands_and_activity(pdb_id, entry):
    if not entry:
        return NA, NA, "no_entry_data"

    ids = (entry.get("rcsb_entry_container_identifiers") or {}).get("non_polymer_entity_ids") or []
    if not ids:
        return NA, NA, "no_nonpolymer_entities_in_structure"

    ligand_names = []
    activity_entries = []
    any_excluded = False

    for eid in ids:
        ejson = safe_get_json(RCSB_NONPOLY_URL.format(pdb_id=pdb_id, entity_id=eid),
                               context=f"RCSB nonpolymer_entity {pdb_id}_{eid}")
        if not ejson:
            continue
        nonpoly = ejson.get("pdbx_entity_nonpoly") or {}
        comp_id = nonpoly.get("comp_id", "")
        name = nonpoly.get("name") or (ejson.get("rcsb_nonpolymer_entity") or {}).get("pdbx_description")

        if comp_id.upper() in NON_INFORMATIVE_COMP_IDS:
            any_excluded = True
            continue

        display_name = name or comp_id or "UNKNOWN_LIGAND"
        ligand_names.append(display_name)

        affinities = ejson.get("rcsb_binding_affinity") or []
        for aff in affinities:
            a_type = aff.get("type", "?")
            a_val = aff.get("value", "?")
            a_unit = aff.get("unit", "")
            a_src = aff.get("provenance_source", "unknown source")
            activity_entries.append(f"{display_name}: {a_type} {a_val} {a_unit} ({a_src})")

    if not ligand_names:
        note = "only_water_ions_or_crystallization_additives_present" if any_excluded else "no_relevant_ligands_found"
        return NA, NA, note

    ligand_col = "; ".join(dict.fromkeys(ligand_names))
    activity_col = "; ".join(activity_entries) if activity_entries else NA
    return ligand_col, activity_col, "rcsb_nonpolymer_entity + rcsb_binding_affinity"

print("Ligand / activity extractor defined")


Ligand / activity extractor defined


In [29]:
rows = []
processed_ok, processed_failed = 0, 0
missing_validation, missing_ligand, missing_activity = 0, 0, 0

retrieval_timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

for pdb_id in pdb_ids:
    entry = entry_json_by_pdb.get(pdb_id)
    row_status = "OK" if entry else "FAILED: RCSB entry fetch failed"

    method, method_note = get_method(entry)
    resolution, res_note = get_resolution(entry)
    deposited, modeled, res_diff, res_note2 = get_residue_counts(pdb_id, entry, chosen_accession)
    r_free_dcc, r_work_dcc, r_diff, dcc_note = get_r_values_dcc(entry)
    vmetrics = get_validation_metrics(entry)
    ligand_name, ligand_activity, ligand_note = get_ligands_and_activity(pdb_id, entry)

    if entry is None:
        processed_failed += 1
    else:
        processed_ok += 1

    if all(vmetrics[k] == NA for k in ["r_free", "clashscore", "rama_outliers", "sidechain_outliers", "rsrz_outliers"]):
        missing_validation += 1
    if ligand_name == NA:
        missing_ligand += 1
    if ligand_activity == NA:
        missing_activity += 1

    rows.append({
        "Protein Name": protein_name_column_value,
        "PDB ID": pdb_id,
        "PDB Link": rcsb_entry_link(pdb_id),
        "Method": method,
        "Resolution (A)": resolution,
        "Deposited Residue Count": deposited,
        "Modeled Residue Count": modeled,
        "Residue Count Difference": res_diff,
        "R-Value Free (DCC)": r_free_dcc,
        "R-Value Work (DCC)": r_work_dcc,
        "R-Value Difference": r_diff,
        "R-free": vmetrics["r_free"],
        "Clashscore": vmetrics["clashscore"],
        "Ramachandran Outliers (%)": vmetrics["rama_outliers"],
        "Sidechain Outliers (%)": vmetrics["sidechain_outliers"],
        "RSRZ Outliers (%)": vmetrics["rsrz_outliers"],
        "Average B, all atoms (A^2)": vmetrics["avg_b_all_atoms"],
        "Ligand Name": ligand_name,
        "Ligand Activity": ligand_activity,
        # helper / traceability columns (kept, per instructions this is allowed):
        "_row_status": row_status,
        "_residue_count_rule_note": res_note2,
        "_r_value_note": dcc_note,
        "_validation_notes": json.dumps(vmetrics.get("_notes", {})),
        "_ligand_note": ligand_note,
        "_retrieved_at": retrieval_timestamp,
    })

df = pd.DataFrame(rows)
print(f"Processed {len(df)} PDB rows. OK={processed_ok}  Failed={processed_failed}")
df.head()


Processed 37 PDB rows. OK=37  Failed=0


,Protein Name,PDB ID,PDB Link,Method,Resolution (A),Deposited Residue Count,Modeled Residue Count,Residue Count Difference,R-Value Free (DCC),R-Value Work (DCC),...,RSRZ Outliers (%),"Average B, all atoms (A^2)",Ligand Name,Ligand Activity,_row_status,_residue_count_rule_note,_r_value_note,_validation_notes,_ligand_note,_retrieved_at
0,ITK (Q08881),1SM2,https://www.rcsb.org/structure/1SM2,X-RAY DIFFRACTION,2.3,264,245,19,N/A,N/A,...,N/A,N/A,STAUROSPORINE,N/A,OK,matched_by_uniprot_accession; representative_c...,DCC_R_free_not_available | DCC_R_work_not_avai...,"{""r_free"": ""not_available_in_validation_report...",rcsb_nonpolymer_entity + rcsb_binding_affinity,2026-09-17 00:14:44 UTC
1,ITK (Q08881),1SNU,https://www.rcsb.org/structure/1SNU,X-RAY DIFFRACTION,2.5,264,245,19,N/A,N/A,...,N/A,N/A,STAUROSPORINE,N/A,OK,matched_by_uniprot_accession; representative_c...,DCC_R_free_not_available | DCC_R_work_not_avai...,"{""r_free"": ""not_available_in_validation_report...",rcsb_nonpolymer_entity + rcsb_binding_affinity,2026-09-17 00:14:44 UTC
2,ITK (Q08881),1SNX,https://www.rcsb.org/structure/1SNX,X-RAY DIFFRACTION,3.2,264,245,19,N/A,N/A,...,N/A,N/A,N/A,N/A,OK,matched_by_uniprot_accession; representative_c...,DCC_R_free_not_available | DCC_R_work_not_avai...,"{""r_free"": ""not_available_in_validation_report...",no_nonpolymer_entities_in_structure,2026-09-17 00:14:44 UTC
3,ITK (Q08881),2E6I,https://www.rcsb.org/structure/2E6I,SOLUTION NMR,N/A,64,64,0,N/A,N/A,...,N/A,N/A,N/A,N/A,OK,matched_by_uniprot_accession; representative_c...,DCC_R_free_not_available | DCC_R_work_not_avai...,"{""r_free"": ""not_available_in_validation_report...",only_water_ions_or_crystallization_additives_p...,2026-09-17 00:14:44 UTC
4,ITK (Q08881),2LMJ,https://www.rcsb.org/structure/2LMJ,SOLUTION NMR,N/A,66,66,0,N/A,N/A,...,N/A,N/A,N/A,N/A,OK,matched_by_uniprot_accession; representative_c...,DCC_R_free_not_available | DCC_R_work_not_avai...,"{""r_free"": ""not_available_in_validation_report...",no_nonpolymer_entities_in_structure,2026-09-17 00:14:44 UTC


In [31]:
def is_number(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool)

validation_report = []

def check(label, condition_series):
    n_fail = int((~condition_series).sum())
    status = "PASS" if n_fail == 0 else f"FAIL ({n_fail} rows)"
    validation_report.append((label, status))
    return n_fail

check("Every row has a Protein Name", df["Protein Name"].astype(bool))
check("Every row has a valid (non-empty) PDB ID", df["PDB ID"].str.len() == 4)
check("Every PDB hyperlink points to rcsb.org/structure/",
      df["PDB Link"].str.startswith("https://www.rcsb.org/structure/"))
check("Deposited Residue Count is numeric or N/A",
      df["Deposited Residue Count"].apply(lambda x: x == NA or is_number(x)))
check("Modeled Residue Count is numeric or N/A",
      df["Modeled Residue Count"].apply(lambda x: x == NA or is_number(x)))
check("Residue Count Difference correctly calculated",
      df.apply(lambda r: r["Residue Count Difference"] == NA or
                (is_number(r["Deposited Residue Count"]) and is_number(r["Modeled Residue Count"]) and
                 r["Residue Count Difference"] == r["Deposited Residue Count"] - r["Modeled Residue Count"]), axis=1))
check("R-Value Difference correctly calculated",
      df.apply(lambda r: r["R-Value Difference"] == NA or
                (is_number(r["R-Value Free (DCC)"]) and is_number(r["R-Value Work (DCC)"]) and
                 abs(r["R-Value Difference"] - (r["R-Value Free (DCC)"] - r["R-Value Work (DCC)"])) < 1e-9), axis=1))
check("No duplicate PDB ID rows", ~df["PDB ID"].duplicated())

print("=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)
for label, status in validation_report:
    print(f"  [{status}] {label}")
print("=" * 60)

any_fail = any("FAIL" in status for _, status in validation_report)
if any_fail:
    print("\nOne or more validation checks FAILED. Review the rows above before trusting the "
          "workbook; the notebook will still export it so you can inspect the failing rows.")
else:
    print("\nAll validation checks passed.")


VALIDATION SUMMARY
  [PASS] Every row has a Protein Name
  [PASS] Every row has a valid (non-empty) PDB ID
  [PASS] Every PDB hyperlink points to rcsb.org/structure/
  [PASS] Deposited Residue Count is numeric or N/A
  [PASS] Modeled Residue Count is numeric or N/A
  [PASS] Residue Count Difference correctly calculated
  [PASS] R-Value Difference correctly calculated
  [PASS] No duplicate PDB ID rows

All validation checks passed.


In [33]:
MAIN_COLUMNS = [
    "Protein Name", "PDB ID", "PDB Link", "Method", "Resolution (A)",
    "Deposited Residue Count", "Modeled Residue Count", "Residue Count Difference",
    "R-Value Free (DCC)", "R-Value Work (DCC)", "R-Value Difference",
    "R-free", "Clashscore", "Ramachandran Outliers (%)", "Sidechain Outliers (%)",
    "RSRZ Outliers (%)", "Average B, all atoms (A^2)", "Ligand Name", "Ligand Activity",
]

main_df = df[MAIN_COLUMNS].copy()

diagnostics_df = df[[
    "PDB ID", "_row_status", "_residue_count_rule_note", "_r_value_note",
    "_validation_notes", "_ligand_note", "_retrieved_at",
]].copy()

sources_rows = []
for _, r in df.iterrows():
    sources_rows.append({
        "PDB ID": r["PDB ID"],
        "UniProt accession": chosen_accession,
        "UniProt entry URL": UNIPROT_ENTRY_URL,
        "UniProt API (PDB xrefs)": f"{UNIPROT_BASE}/{chosen_accession}.json",
        "RCSB entry URL": r["PDB Link"],
        "RCSB Data API (entry)": RCSB_ENTRY_URL.format(pdb_id=r["PDB ID"]),
        "wwPDB validation source": "pdbx_vrpt_summary field within the RCSB entry record above "
                                    "(RCSB's integration of the wwPDB Validation Report)",
        "Ligand/activity source": "rcsb_nonpolymer_entity + rcsb_binding_affinity via RCSB Data API",
        "Retrieved at (UTC)": r["_retrieved_at"],
    })
sources_df = pd.DataFrame(sources_rows)

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    main_df.to_excel(writer, sheet_name="PDB_Data", index=False)
    sources_df.to_excel(writer, sheet_name="Sources", index=False)
    diagnostics_df.to_excel(writer, sheet_name="Diagnostics", index=False)

print(f"Wrote {OUTPUT_XLSX} with sheets: PDB_Data, Sources, Diagnostics")


Wrote Protein_PDB_Data.xlsx with sheets: PDB_Data, Sources, Diagnostics


In [35]:
# --- Formatting pass: hyperlinks, freeze panes, filters, column widths, number formats ---
wb = load_workbook(OUTPUT_XLSX)
ws = wb["PDB_Data"]

header_font = Font(bold=True, color="FFFFFF")
header_fill = PatternFill(start_color="305496", end_color="305496", fill_type="solid")

for col_idx, col_name in enumerate(MAIN_COLUMNS, start=1):
    cell = ws.cell(row=1, column=col_idx)
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

ws.freeze_panes = "A2"
ws.auto_filter.ref = ws.dimensions

pdb_id_col = MAIN_COLUMNS.index("PDB ID") + 1
pdb_link_col = MAIN_COLUMNS.index("PDB Link") + 1

numeric_cols = {
    "Resolution (A)", "Deposited Residue Count", "Modeled Residue Count",
    "Residue Count Difference", "R-Value Free (DCC)", "R-Value Work (DCC)",
    "R-Value Difference", "R-free", "Clashscore", "Ramachandran Outliers (%)",
    "Sidechain Outliers (%)", "RSRZ Outliers (%)", "Average B, all atoms (A^2)",
}
numeric_col_idx = {MAIN_COLUMNS.index(c) + 1 for c in numeric_cols}

for row_idx in range(2, ws.max_row + 1):
    pdb_id_val = ws.cell(row=row_idx, column=pdb_id_col).value
    link_cell = ws.cell(row=row_idx, column=pdb_link_col)
    if pdb_id_val:
        link_cell.hyperlink = f"https://www.rcsb.org/structure/{pdb_id_val}"
        link_cell.value = pdb_id_val          # displayed text = PDB ID, per requirement
        link_cell.font = Font(color="0563C1", underline="single")

    for c_idx in numeric_col_idx:
        cell = ws.cell(row=row_idx, column=c_idx)
        if isinstance(cell.value, str) and cell.value != NA:
            try:
                cell.value = float(cell.value)
            except ValueError:
                pass
        if isinstance(cell.value, (int, float)):
            cell.number_format = "0.###"

for col_idx, col_name in enumerate(MAIN_COLUMNS, start=1):
    max_len = max(
        [len(str(col_name))] +
        [len(str(ws.cell(row=r, column=col_idx).value)) for r in range(2, ws.max_row + 1)]
    )
    ws.column_dimensions[get_column_letter(col_idx)].width = min(max(max_len + 2, 10), 45)

# Light formatting for the other two sheets too (freeze header, autosize, filter).
for sheet_name in ("Sources", "Diagnostics"):
    s = wb[sheet_name]
    s.freeze_panes = "A2"
    s.auto_filter.ref = s.dimensions
    for col_idx in range(1, s.max_column + 1):
        max_len = max(
            [len(str(s.cell(row=1, column=col_idx).value or ""))] +
            [len(str(s.cell(row=r, column=col_idx).value or "")) for r in range(2, min(s.max_row, 200) + 1)]
        )
        s.column_dimensions[get_column_letter(col_idx)].width = min(max(max_len + 2, 10), 60)
    for col_idx in range(1, s.max_column + 1):
        c = s.cell(row=1, column=col_idx)
        c.font = header_font
        c.fill = header_fill

wb.save(OUTPUT_XLSX)
print(f"Formatting complete -> {OUTPUT_XLSX}")


Formatting complete -> Protein_PDB_Data.xlsx


In [37]:
print("=" * 60)
print("PROCESSING SUMMARY")
print("=" * 60)
print(f"Protein searched:            {protein_name}")
print(f"UniProt accession:           {chosen_accession}")
print(f"Total PDB IDs found:         {len(pdb_ids)}")
print(f"Successfully processed:      {processed_ok}")
print(f"Failed:                      {processed_failed}")
print(f"Missing validation data:     {missing_validation}")
print(f"Missing ligand data:         {missing_ligand}")
print(f"Missing activity data:       {missing_activity}")
print("=" * 60)

if FAILURE_LOG:
    print(f"\n{len(FAILURE_LOG)} individual request failures were logged during this run:")
    for f in FAILURE_LOG:
        print(f"  - [{f['context']}] {f['reason']}  ({f['url']})")
else:
    print("\nNo individual request failures were logged.")

print(f"\nWorkbook written to: {OUTPUT_XLSX}")


PROCESSING SUMMARY
Protein searched:            ITK
UniProt accession:           Q08881
Total PDB IDs found:         37
Successfully processed:      37
Failed:                      0
Missing validation data:     0
Missing ligand data:         4
Missing activity data:       37

No individual request failures were logged.

Workbook written to: Protein_PDB_Data.xlsx
